# Bank Customer Churn Dashboard
This notebook builds a dark-theme interactive dashboard as a standalone HTML file for the **Bank Customer Churn** dataset.

**What it does**
- loads the CSV
- validates and renames columns if needed
- creates KPI cards and interactive charts
- exports a standalone dashboard HTML file
- shows an inline preview link at the end

Keep the CSV in the same folder as this notebook, or update the `DATA_PATH` cell.


In [ ]:
# Core imports
import pandas as pd
import numpy as np
import json
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from pathlib import Path


In [ ]:
# File path
DATA_PATH = Path("Bank Customer Churn Prediction.csv")
OUTPUT_HTML = Path("bank_churn_dashboard.html")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Keep the CSV in the same folder as the notebook or update DATA_PATH."
    )

df_raw = pd.read_csv(DATA_PATH)
print("Dataset shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
df_raw.head()


Dataset shape: (10000, 12)
Columns: ['customer_id', 'credit_score', 'country', 'gender', 'age', 'tenure', 'balance', 'products_number', 'credit_card', 'active_member', 'estimated_salary', 'churn']


,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
# Standardise column names and validate expected fields
rename_map = {
    "customer_id": "customer_id",
    "credit_score": "credit_score",
    "country": "country",
    "gender": "gender",
    "age": "age",
    "tenure": "tenure",
    "balance": "balance",
    "products_number": "products_number",
    "credit_card": "credit_card",
    "active_member": "active_member",
    "estimated_salary": "estimated_salary",
    "churn": "churn",
    # fallback aliases in case someone uses a slightly different version
    "CustomerId": "customer_id",
    "CreditScore": "credit_score",
    "Geography": "country",
    "Gender": "gender",
    "Age": "age",
    "Tenure": "tenure",
    "Balance": "balance",
    "NumOfProducts": "products_number",
    "HasCrCard": "credit_card",
    "IsActiveMember": "active_member",
    "EstimatedSalary": "estimated_salary",
    "Exited": "churn",
}

df = df_raw.rename(columns={c: rename_map.get(c, c) for c in df_raw.columns}).copy()

required = [
    "credit_score", "country", "gender", "age", "tenure", "balance",
    "products_number", "credit_card", "active_member", "estimated_salary", "churn"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# clean types
numeric_cols = ["credit_score","age","tenure","balance","products_number","credit_card","active_member","estimated_salary","churn"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=required).copy()
df["churn"] = df["churn"].astype(int)

# friendly labels
df["churn_label"] = df["churn"].map({0: "Retained", 1: "Churned"})
df["credit_card_label"] = df["credit_card"].map({0: "No Card", 1: "Has Card"})
df["active_member_label"] = df["active_member"].map({0: "Inactive", 1: "Active"})

# age bands for cleaner presentation
df["age_band"] = pd.cut(
    df["age"],
    bins=[17, 25, 35, 45, 55, 65, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66+"],
    include_lowest=True
)

print("Cleaned shape:", df.shape)
df.head()


Cleaned shape: (10000, 16)


,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,churn_label,credit_card_label,active_member_label,age_band
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1,Churned,Has Card,Active,36-45
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,Retained,No Card,Active,36-45
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,Churned,Has Card,Inactive,36-45
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0,Retained,No Card,Inactive,36-45
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,Retained,Has Card,Active,36-45


In [ ]:
# Theme settings based on the dark PPT look
BG = "#071521"
CARD = "#0e2233"
CARD2 = "#122b40"
GRID = "rgba(255,255,255,0.10)"
TEXT = "#EAF6FF"
MUTED = "#A8C0D8"
CYAN = "#00D4FF"
BLUE = "#4DA3FF"
ORANGE = "#FF8A3D"

plotly_template = dict(
    layout=dict(
        paper_bgcolor=CARD,
        plot_bgcolor=CARD,
        font=dict(color=TEXT, family="Arial"),
        title=dict(font=dict(size=20, color=TEXT)),
        xaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID, tickfont=dict(color=TEXT), title_font=dict(color=TEXT)),
        yaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID, tickfont=dict(color=TEXT), title_font=dict(color=TEXT)),
        legend=dict(font=dict(color=TEXT), bgcolor="rgba(0,0,0,0)")
    )
)

def style_fig(fig, height=420):
    fig.update_layout(template=plotly_template, margin=dict(l=40, r=30, t=60, b=40), height=height)
    return fig


In [ ]:
# KPI calculations
total_customers = len(df)
churned_customers = int(df["churn"].sum())
retained_customers = total_customers - churned_customers
churn_rate = df["churn"].mean()
avg_balance = df["balance"].mean()
avg_credit_score = df["credit_score"].mean()
avg_salary = df["estimated_salary"].mean()

kpis = {
    "Total Customers": f"{total_customers:,}",
    "Churned Customers": f"{churned_customers:,}",
    "Churn Rate": f"{churn_rate:.1%}",
    "Avg Balance": f"€{avg_balance:,.0f}",
    "Avg Credit Score": f"{avg_credit_score:.0f}",
    "Avg Estimated Salary": f"€{avg_salary:,.0f}",
}
kpis


{'Total Customers': '10,000',
 'Churned Customers': '2,037',
 'Churn Rate': '20.4%',
 'Avg Balance': '€76,486',
 'Avg Credit Score': '651',
 'Avg Estimated Salary': '€100,090'}

In [ ]:
# Aggregations for charts
churn_dist = df["churn_label"].value_counts().reset_index()
churn_dist.columns = ["status", "count"]

country_churn = (
    df.groupby("country")["churn"]
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .reset_index(name="churn_rate_pct")
)

tenure_churn = (
    df.groupby("tenure")["churn"]
    .mean()
    .mul(100)
    .reset_index(name="churn_rate_pct")
    .sort_values("tenure")
)

product_churn = (
    df.groupby("products_number")["churn"]
    .mean()
    .mul(100)
    .reset_index(name="churn_rate_pct")
    .sort_values("products_number")
)

ageband_churn = (
    df.groupby("age_band", observed=False)["churn"]
    .mean()
    .mul(100)
    .reset_index(name="churn_rate_pct")
)

balance_by_churn = df[["churn_label", "balance"]].copy()

corr_cols = ["credit_score","age","tenure","balance","products_number","credit_card","active_member","estimated_salary","churn"]
corr = df[corr_cols].corr(numeric_only=True)

country_churn, tenure_churn.head(), product_churn.head()


(   country  churn_rate_pct
 0  Germany       32.443204
 1    Spain       16.673395
 2   France       16.154767,
    tenure  churn_rate_pct
 0       0       23.002421
 1       1       22.415459
 2       2       19.179389
 3       3       21.110010
 4       4       20.525784,
    products_number  churn_rate_pct
 0                1       27.714398
 1                2        7.581699
 2                3       82.706767
 3                4      100.000000)

In [ ]:
# Build interactive charts

# 1. Churn distribution donut
fig1 = px.pie(
    churn_dist,
    names="status",
    values="count",
    hole=0.62,
    color="status",
    color_discrete_map={"Retained": CYAN, "Churned": ORANGE},
    title="Customer Base Overview: Retained vs Churned"
)
fig1.update_traces(
    textposition="inside",
    textinfo="percent+label",
    marker=dict(line=dict(color=BG, width=2))
)
fig1.add_annotation(
    text=f"<b>{churn_rate:.1%}</b><br>churn rate",
    x=0.5, y=0.5, showarrow=False,
    font=dict(size=22, color=TEXT)
)
style_fig(fig1, 430)

# 2. Churn by country
fig2 = px.bar(
    country_churn,
    x="country", y="churn_rate_pct",
    color="country",
    title="Churn Rate by Country",
    color_discrete_sequence=[ORANGE, BLUE, CYAN]
)
fig2.update_layout(showlegend=False, yaxis_title="Churn Rate (%)", xaxis_title="")
style_fig(fig2, 430)

# 3. Tenure vs churn
fig3 = px.line(
    tenure_churn,
    x="tenure", y="churn_rate_pct",
    markers=True,
    title="Churn Risk Across Customer Tenure"
)
fig3.update_traces(line=dict(color=CYAN, width=3), marker=dict(size=9, color=ORANGE))
fig3.update_layout(yaxis_title="Churn Rate (%)", xaxis_title="Tenure (Years)")
style_fig(fig3, 430)

# 4. Products vs churn
fig4 = px.bar(
    product_churn,
    x="products_number", y="churn_rate_pct",
    text_auto=".1f",
    title="Churn Rate by Number of Products"
)
fig4.update_traces(marker_color=ORANGE)
fig4.update_layout(yaxis_title="Churn Rate (%)", xaxis_title="Number of Products")
style_fig(fig4, 430)

# 5. Age band churn
fig5 = px.bar(
    ageband_churn,
    x="age_band", y="churn_rate_pct",
    text_auto=".1f",
    title="Churn Rate by Age Band"
)
fig5.update_traces(marker_color=BLUE)
fig5.update_layout(yaxis_title="Churn Rate (%)", xaxis_title="Age Band")
style_fig(fig5, 430)

# 6. Balance vs churn
fig6 = px.box(
    balance_by_churn,
    x="churn_label", y="balance",
    color="churn_label",
    title="Balance Distribution by Churn Status",
    color_discrete_map={"Retained": CYAN, "Churned": ORANGE},
    points="outliers"
)
fig6.update_layout(showlegend=False, xaxis_title="", yaxis_title="Balance")
style_fig(fig6, 430)

# 7. Correlation heatmap
fig7 = go.Figure(
    data=go.Heatmap(
        z=corr.values,
        x=corr.columns,
        y=corr.index,
        colorscale=[
            [0.0, "#16354F"],
            [0.5, "#0FA3C9"],
            [1.0, "#FF8A3D"]
        ],
        zmin=-1, zmax=1,
        text=np.round(corr.values, 2),
        texttemplate="%{text}",
        hovertemplate="X: %{x}<br>Y: %{y}<br>Corr: %{z:.2f}<extra></extra>"
    )
)
fig7.update_layout(title="Feature Correlation Heatmap")
style_fig(fig7, 500)

fig1.show()
fig2.show()
fig3.show()
fig4.show()
fig5.show()
fig6.show()
#fig7.show()


In [ ]:
rename_map = {
    "CustomerId": "customer_id",
    "CreditScore": "credit_score",
    "Geography": "country",
    "Gender": "gender",
    "Age": "age",
    "Tenure": "tenure",
    "Balance": "balance",
    "NumOfProducts": "products_number",
    "HasCrCard": "credit_card",
    "IsActiveMember": "active_member",
    "EstimatedSalary": "estimated_salary",
    "Exited": "churn",
    "customer_id": "customer_id",
    "credit_score": "credit_score",
    "country": "country",
    "gender": "gender",
    "age": "age",
    "tenure": "tenure",
    "balance": "balance",
    "products_number": "products_number",
    "credit_card": "credit_card",
    "active_member": "active_member",
    "estimated_salary": "estimated_salary",
    "churn": "churn",
}

df = df_raw.rename(columns={c: rename_map.get(c, c) for c in df_raw.columns}).copy()

required = [
    "credit_score", "country", "gender", "age", "tenure", "balance",
    "products_number", "credit_card", "active_member", "estimated_salary", "churn"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

df["churn"] = df["churn"].astype(int)
df["country"] = df["country"].astype(str)
df["gender"] = df["gender"].astype(str)

bins = [17, 30, 40, 50, 60, 200]
labels = ["18-30", "31-40", "41-50", "51-60", "60+"]
df["age_band"] = pd.cut(df["age"], bins=bins, labels=labels, include_lowest=True)
df["age_band"] = df["age_band"].astype(str).replace("nan", None)

data_records = df[[
    "country", "gender", "age", "age_band", "tenure", "balance",
    "products_number", "credit_score", "estimated_salary", "churn"
]].replace({np.nan: None}).to_dict(orient="records")

len(data_records), data_records[:2]


(10000,
 [{'country': 'France',
   'gender': 'Female',
   'age': 42,
   'age_band': '41-50',
   'tenure': 2,
   'balance': 0.0,
   'products_number': 1,
   'credit_score': 619,
   'estimated_salary': 101348.88,
   'churn': 1},
  {'country': 'Spain',
   'gender': 'Female',
   'age': 41,
   'age_band': '41-50',
   'tenure': 1,
   'balance': 83807.86,
   'products_number': 1,
   'credit_score': 608,
   'estimated_salary': 112542.58,
   'churn': 0}])

In [ ]:
BG = "#071521"
CARD = "#0e2233"
CARD2 = "#122b40"
GRID = "rgba(255,255,255,0.10)"
TEXT = "#EAF6FF"
MUTED = "#A8C0D8"
CYAN = "#17C3E6"
BLUE = "#4D96FF"
ORANGE = "#FF8A3D"

html_template = '''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Bank Customer Churn Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
:root {
    --bg: __BG__;
    --card: __CARD__;
    --card2: __CARD2__;
    --grid: __GRID__;
    --text: __TEXT__;
    --muted: __MUTED__;
    --cyan: __CYAN__;
    --blue: __BLUE__;
    --orange: __ORANGE__;
    --border: rgba(255,255,255,0.08);
}
* { box-sizing: border-box; }
body {
    margin: 0;
    min-height: 100vh;
    background:
        radial-gradient(circle at top left, rgba(0,212,255,0.10), transparent 28%),
        radial-gradient(circle at top right, rgba(255,138,61,0.09), transparent 25%),
        linear-gradient(180deg, #04101a 0%, var(--bg) 100%);
    color: var(--text);
    font-family: Arial, Helvetica, sans-serif;
}
.container {
    max-width: 1460px;
    margin: 0 auto;
    padding: 18px;
}
.hero {
    background: linear-gradient(90deg, rgba(0,212,255,0.10), rgba(255,138,61,0.08));
    border: 1px solid var(--border);
    border-radius: 28px;
    padding: 24px 30px;
    margin-bottom: 22px;
    box-shadow: 0 14px 40px rgba(0,0,0,0.22);
}
.hero h1 {
    margin: 0 0 10px 0;
    font-size: 38px;
    line-height: 1.12;
    letter-spacing: -0.02em;
}
.hero p {
    margin: 0;
    color: var(--muted);
    font-size: 15px;
    line-height: 1.55;
}
.filter-bar {
    display: flex;
    align-items: center;
    gap: 16px;
    background: var(--card);
    border: 1px solid var(--border);
    border-radius: 20px;
    padding: 16px 18px;
    margin-bottom: 22px;
    flex-wrap: wrap;
    box-shadow: 0 14px 40px rgba(0,0,0,0.18);
}
.filter-label {
    color: var(--muted);
    font-size: 12px;
    font-weight: 700;
    letter-spacing: 0.06em;
    text-transform: uppercase;
}
.filter-icon {
    font-size: 14px;
    opacity: 0.95;
}
.divider {
    width: 1px;
    height: 30px;
    background: var(--border);
}
.filter-select {
    min-width: 150px;
    background: #0a1d2e;
    color: var(--text);
    border: 1px solid rgba(0,212,255,0.32);
    border-radius: 12px;
    padding: 9px 34px 9px 14px;
    font-size: 14px;
    cursor: pointer;
    appearance: none;
    -webkit-appearance: none;
    background-image:
        linear-gradient(45deg, transparent 50%, var(--cyan) 50%),
        linear-gradient(135deg, var(--cyan) 50%, transparent 50%);
    background-position:
        calc(100% - 18px) calc(50% - 1px),
        calc(100% - 12px) calc(50% - 1px);
    background-size: 6px 6px, 6px 6px;
    background-repeat: no-repeat;
}
.filter-select:focus {
    outline: none;
    border-color: var(--cyan);
    box-shadow: 0 0 0 3px rgba(0,212,255,0.10);
}
.active-filter-badge {
    margin-left: auto;
    background: rgba(0,212,255,0.12);
    color: var(--cyan);
    border: 1px solid rgba(0,212,255,0.24);
    border-radius: 999px;
    padding: 6px 14px;
    font-size: 12px;
    font-weight: 700;
    white-space: nowrap;
}
.kpi-grid {
    display: grid;
    grid-template-columns: repeat(6, minmax(0, 1fr));
    gap: 16px;
    margin-bottom: 20px;
}
.kpi {
    background: linear-gradient(180deg, rgba(14,34,51,0.98), rgba(10,28,44,0.98));
    border: 1px solid var(--border);
    border-radius: 22px;
    padding: 16px 18px;
    min-height: 104px;
    box-shadow: 0 14px 40px rgba(0,0,0,0.18);
}
.kpi-label {
    color: var(--muted);
    font-size: 13px;
    margin-bottom: 10px;
}
.kpi-value {
    font-size: 26px;
    font-weight: 800;
    letter-spacing: -0.02em;
}
.grid-2 {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 18px;
    margin-bottom: 18px;
}
.card {
    background: var(--card);
    border: 1px solid var(--border);
    border-radius: 24px;
    padding: 10px;
    box-shadow: 0 14px 40px rgba(0,0,0,0.18);
}
.chart {
    width: 100%;
    height: 500px;
}
.note {
    margin-top: 4px;
    color: var(--muted);
    font-size: 13px;
    line-height: 1.5;
}
@media (max-width: 1280px) {
    .kpi-grid { grid-template-columns: repeat(3, minmax(0,1fr)); }
    .grid-2 { grid-template-columns: 1fr; }
}
@media (max-width: 760px) {
    .kpi-grid { grid-template-columns: repeat(2, minmax(0,1fr)); }
    .hero h1 { font-size: 30px; }
    .active-filter-badge { margin-left: 0; }
}
</style>
</head>
<body>
<div class="container">
    <section class="hero">
        <h1>Bank Customer Churn Dashboard</h1>

    </section>

    <section class="filter-bar">
        <span class="filter-label filter-icon">🔎 Filters</span>
        <div class="divider"></div>
        <span class="filter-label">Country</span>
        <select id="sel-country" class="filter-select"></select>
        <div class="divider"></div>
        <span class="filter-label">Gender</span>
        <select id="sel-gender" class="filter-select"></select>
        <div class="active-filter-badge" id="filter-badge"></div>
    </section>

    <section class="kpi-grid">
        <div class="kpi"><div class="kpi-label">Total Customers</div><div class="kpi-value" id="kpi-total">-</div></div>
        <div class="kpi"><div class="kpi-label">Churned Customers</div><div class="kpi-value" id="kpi-churned">-</div></div>
        <div class="kpi"><div class="kpi-label">Churn Rate</div><div class="kpi-value" id="kpi-rate">-</div></div>
        <div class="kpi"><div class="kpi-label">Avg Balance</div><div class="kpi-value" id="kpi-balance">-</div></div>
        <div class="kpi"><div class="kpi-label">Avg Credit Score</div><div class="kpi-value" id="kpi-credit">-</div></div>
        <div class="kpi"><div class="kpi-label">Avg Estimated Salary</div><div class="kpi-value" id="kpi-salary">-</div></div>
    </section>

    <section class="grid-2">
        <div class="card"><div id="chart-donut" class="chart"></div></div>
        <div class="card"><div id="chart-country" class="chart"></div></div>
    </section>

    <section class="grid-2">
        <div class="card"><div id="chart-tenure" class="chart"></div></div>
        <div class="card"><div id="chart-products" class="chart"></div></div>
    </section>

    <section class="grid-2">
        <div class="card"><div id="chart-age" class="chart"></div></div>
        <div class="card"><div id="chart-balance" class="chart"></div></div>
    </section>

</div>

<script>
const DATA = __DATA_JSON__;

const COLORS = {
    bg: "__BG__",
    card: "__CARD__",
    card2: "__CARD2__",
    grid: "__GRID__",
    text: "__TEXT__",
    muted: "__MUTED__",
    cyan: "__CYAN__",
    blue: "__BLUE__",
    orange: "__ORANGE__"
};

const countrySelect = document.getElementById("sel-country");
const genderSelect = document.getElementById("sel-gender");
const badge = document.getElementById("filter-badge");

function uniqueValues(arr, key) {
    const vals = [...new Set(arr.map(d => d[key]).filter(v => v !== null && v !== undefined && v !== "" && v !== "nan"))];
    vals.sort();
    return vals;
}

function fillSelect(selectEl, allLabel, values) {
    selectEl.innerHTML = "";
    const allOpt = document.createElement("option");
    allOpt.value = "All";
    allOpt.textContent = allLabel;
    selectEl.appendChild(allOpt);

    values.forEach(v => {
        const opt = document.createElement("option");
        opt.value = v;
        opt.textContent = v;
        selectEl.appendChild(opt);
    });
}

function fmtInt(x) {
    return new Intl.NumberFormat().format(x);
}
function fmtMoney(x) {
    return "€" + new Intl.NumberFormat(undefined, {maximumFractionDigits: 0}).format(x);
}
function fmtPct(x) {
    return (x * 100).toFixed(1) + "%";
}

function baseLayout(title) {
    return {
        title: {text: title, x: 0.05, font: {size: 20, color: COLORS.text}},
        paper_bgcolor: COLORS.card,
        plot_bgcolor: COLORS.card,
        font: {family: "Arial, Helvetica, sans-serif", color: COLORS.text},
        margin: {l: 70, r: 20, t: 54, b: 60},
        legend: {orientation: "v", font: {color: COLORS.text}, bgcolor: "rgba(0,0,0,0)"},
        xaxis: {
            gridcolor: COLORS.grid,
            zerolinecolor: COLORS.grid,
            linecolor: COLORS.grid,
            tickfont: {color: COLORS.text},
            titlefont: {color: COLORS.text}
        },
        yaxis: {
            gridcolor: COLORS.grid,
            zerolinecolor: COLORS.grid,
            linecolor: COLORS.grid,
            tickfont: {color: COLORS.text},
            titlefont: {color: COLORS.text}
        }
    };
}

function groupMean(rows, key) {
    const buckets = {};
    rows.forEach(r => {
        const k = r[key];
        if (k === null || k === undefined || k === "" || k === "nan") return;
        if (!buckets[k]) buckets[k] = {sum: 0, count: 0};
        buckets[k].sum += Number(r.churn || 0);
        buckets[k].count += 1;
    });
    return Object.keys(buckets).map(k => ({
        key: k,
        value: buckets[k].count ? (buckets[k].sum / buckets[k].count) * 100 : 0
    }));
}

function filterRows() {
    const country = countrySelect.value;
    const gender = genderSelect.value;
    return DATA.filter(d =>
        (country === "All" || d.country === country) &&
        (gender === "All" || d.gender === gender)
    );
}

function updateKPIs(rows) {
    const n = rows.length;
    const churned = rows.reduce((s, d) => s + Number(d.churn || 0), 0);
    const churnRate = n ? churned / n : 0;
    const avgBalance = n ? rows.reduce((s, d) => s + Number(d.balance || 0), 0) / n : 0;
    const avgCredit = n ? rows.reduce((s, d) => s + Number(d.credit_score || 0), 0) / n : 0;
    const avgSalary = n ? rows.reduce((s, d) => s + Number(d.estimated_salary || 0), 0) / n : 0;

    document.getElementById("kpi-total").textContent = fmtInt(n);
    document.getElementById("kpi-churned").textContent = fmtInt(churned);
    document.getElementById("kpi-rate").textContent = fmtPct(churnRate);
    document.getElementById("kpi-balance").textContent = fmtMoney(avgBalance);
    document.getElementById("kpi-credit").textContent = avgCredit.toFixed(0);
    document.getElementById("kpi-salary").textContent = fmtMoney(avgSalary);

    const countryText = countrySelect.value === "All" ? "All" : countrySelect.value;
    const genderText = genderSelect.value === "All" ? "All" : genderSelect.value;
    badge.textContent = `Country: ${countryText} | Gender: ${genderText} | ${fmtInt(n)} customers`;
}

function drawDonut(rows) {
    const churned = rows.reduce((s, d) => s + Number(d.churn || 0), 0);
    const retained = rows.length - churned;
    const rate = rows.length ? churned / rows.length : 0;

    const trace = {
        type: "pie",
        labels: ["Retained", "Churned"],
        values: [retained, churned],
        hole: 0.62,
        textinfo: "percent+label",
        sort: false,
        marker: {colors: [COLORS.cyan, COLORS.orange], line: {color: COLORS.bg, width: 2}}
    };
    const layout = baseLayout("Customer Base Overview: Retained vs Churned");
    layout.annotations = [{
        text: `<b>${fmtPct(rate)}</b><br>churn rate`,
        x: 0.5, y: 0.5, showarrow: false,
        font: {size: 20, color: COLORS.text}
    }];
    layout.showlegend = true;
    Plotly.react("chart-donut", [trace], layout, {responsive: true});
}

function drawCountry(rows) {
    let agg = groupMean(rows, "country").sort((a, b) => b.value - a.value);
    const trace = {
        type: "bar",
        x: agg.map(d => d.key),
        y: agg.map(d => d.value),
        marker: {color: [COLORS.orange, COLORS.blue, COLORS.cyan]},
        hovertemplate: "country=%{x}<br>churn_rate_pct=%{y:.4f}<extra></extra>"
    };
    const layout = baseLayout("Churn Rate by Country");
    layout.yaxis.title = "Churn Rate (%)";
    Plotly.react("chart-country", [trace], layout, {responsive: true});
}

function drawTenure(rows) {
    let agg = groupMean(rows, "tenure").sort((a, b) => Number(a.key) - Number(b.key));
    const trace = {
        type: "scatter",
        mode: "lines+markers",
        x: agg.map(d => d.key),
        y: agg.map(d => d.value),
        line: {color: COLORS.cyan, width: 3},
        marker: {size: 8, color: COLORS.cyan},
        hovertemplate: "tenure=%{x}<br>churn_rate_pct=%{y:.4f}<extra></extra>"
    };
    const layout = baseLayout("Churn Rate by Tenure");
    layout.xaxis.title = "Tenure";
    layout.yaxis.title = "Churn Rate (%)";
    Plotly.react("chart-tenure", [trace], layout, {responsive: true});
}

function drawProducts(rows) {
    let agg = groupMean(rows, "products_number").sort((a, b) => Number(a.key) - Number(b.key));
    const trace = {
        type: "bar",
        x: agg.map(d => d.key),
        y: agg.map(d => d.value),
        marker: {color: COLORS.orange},
        hovertemplate: "products=%{x}<br>churn_rate_pct=%{y:.4f}<extra></extra>"
    };
    const layout = baseLayout("Churn Rate by Number of Products");
    layout.xaxis.title = "Products";
    layout.yaxis.title = "Churn Rate (%)";
    Plotly.react("chart-products", [trace], layout, {responsive: true});
}

function drawAge(rows) {
    const order = ["18-30", "31-40", "41-50", "51-60", "60+"];
    let agg = groupMean(rows, "age_band");
    agg = agg.filter(d => order.includes(d.key)).sort((a, b) => order.indexOf(a.key) - order.indexOf(b.key));
    const trace = {
        type: "bar",
        x: agg.map(d => d.key),
        y: agg.map(d => d.value),
        marker: {color: COLORS.blue},
        hovertemplate: "age_band=%{x}<br>churn_rate_pct=%{y:.4f}<extra></extra>"
    };
    const layout = baseLayout("Churn Rate by Age Band");
    layout.xaxis.title = "Age Band";
    layout.yaxis.title = "Churn Rate (%)";
    Plotly.react("chart-age", [trace], layout, {responsive: true});
}

function drawBalance(rows) {
    const retained = rows.filter(d => Number(d.churn) === 0).map(d => Number(d.balance || 0));
    const churned = rows.filter(d => Number(d.churn) === 1).map(d => Number(d.balance || 0));
    const traces = [
        {type: "box", name: "Retained", y: retained, marker: {color: COLORS.cyan}, line: {color: COLORS.cyan}, boxpoints: false},
        {type: "box", name: "Churned", y: churned, marker: {color: COLORS.orange}, line: {color: COLORS.orange}, boxpoints: false}
    ];
    const layout = baseLayout("Balance Distribution by Churn Status");
    layout.yaxis.title = "Balance";
    Plotly.react("chart-balance", traces, layout, {responsive: true});
}

function renderAll() {
    const rows = filterRows();
    updateKPIs(rows);
    drawDonut(rows);
    drawCountry(rows);
    drawTenure(rows);
    drawProducts(rows);
    drawAge(rows);
    drawBalance(rows);
}

fillSelect(countrySelect, "All Countries", uniqueValues(DATA, "country"));
fillSelect(genderSelect, "All Genders", uniqueValues(DATA, "gender"));

countrySelect.addEventListener("change", renderAll);
genderSelect.addEventListener("change", renderAll);

renderAll();
</script>
</body>
</html>
'''

dashboard_html = html_template
for key, value in {
    "__BG__": BG,
    "__CARD__": CARD,
    "__CARD2__": CARD2,
    "__GRID__": GRID,
    "__TEXT__": TEXT,
    "__MUTED__": MUTED,
    "__CYAN__": CYAN,
    "__BLUE__": BLUE,
    "__ORANGE__": ORANGE,
    "__DATA_JSON__": json.dumps(data_records),
}.items():
    dashboard_html = dashboard_html.replace(key, value)

OUTPUT_HTML.write_text(dashboard_html, encoding="utf-8")
print(f"Dashboard saved to: {OUTPUT_HTML.resolve()}")

Dashboard saved to: /Users/debasmitbora/Downloads/bank_churn_dashboard.html
